# 01 — Preprocessing and Sequence Building

This notebook prepares the PhysioNet/CinC 2019 Sepsis Prediction data for sequential pattern mining. It loads patient-level PSV files from one selected training set, performs patient-level preprocessing, identifies sepsis onset, constructs positive pre-sepsis and negative comparison windows, and converts clinical measurements into symbolic sequences.

**Important:** This notebook is intentionally the first implementation stage. It does not perform PrefixSpan, classification, or evaluation.

## 1. Configuration

Change only the paths and parameters below. Keep the raw dataset outside Git tracking.

In [ ]:
from pathlib import Path
import pickle
import random
import numpy as np
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Use ONE predefined training set for this implementation.
# Change to "training_setB" if that is the set you choose.
TRAINING_SET = "training_setA"
INPUT_DIR = DATA_DIR / TRAINING_SET

# Output directory
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "sequences"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sequence construction parameters
PRE_SEPSIS_HOURS = 6
NEGATIVE_WINDOW_HOURS = PRE_SEPSIS_HOURS

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


## 2. Locate Patient PSV Files

In [ ]:
if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {INPUT_DIR}\n"
        "Update DATA_DIR/TRAINING_SET in the configuration cell."
    )

psv_files = sorted(INPUT_DIR.glob("*.psv"))
if not psv_files:
    raise FileNotFoundError(f"No .psv files found in {INPUT_DIR}")

print(f"Found {len(psv_files):,} patient PSV files.")
print("Example files:")
for path in psv_files[:5]:
    print(" -", path.name)


## 3. Load a Patient and Inspect the Raw Structure

PSV files are pipe-separated. Missing measurements are expected and are represented as missing values.

In [ ]:
sample_df = pd.read_csv(psv_files[0], sep="|")
print("Shape:", sample_df.shape)
print("Columns:")
print(sample_df.columns.tolist())

display(sample_df.head())


## 4. Clinical Variables Used for Symbolic Sequences

The exact available columns are taken from the downloaded dataset. This implementation prefers clinically meaningful physiological/laboratory variables and excludes `SepsisLabel`, `ICULOS`, and `HospAdmTime` from the symbolic event vocabulary.

The cell below automatically selects available variables from a conservative candidate list. Do not silently invent columns.

In [ ]:
CANDIDATE_FEATURES = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2",
    "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN",
    "Alkalinephos", "Calcium", "Chloride", "Creatinine", "Bilirubin_direct",
    "Glucose", "Lactate", "Magnesium", "Phosphate", "Potassium", "Bilirubin_total",
    "TroponinI", "Hct", "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets"
]

AVAILABLE_FEATURES = [c for c in CANDIDATE_FEATURES if c in sample_df.columns]
print(f"Available clinical features selected: {len(AVAILABLE_FEATURES)}")
print(AVAILABLE_FEATURES)

if not AVAILABLE_FEATURES:
    raise ValueError("No candidate clinical features were found in the dataset.")


## 5. Basic Patient-Level Cleaning

Keep the original hourly order. Convert selected clinical variables to numeric values and preserve missing observations. No forward-looking information is introduced here.

In [ ]:
def load_patient(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="|")
    for col in AVAILABLE_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    if "SepsisLabel" not in df.columns:
        raise ValueError(f"SepsisLabel missing from {path.name}")
    df["SepsisLabel"] = pd.to_numeric(df["SepsisLabel"], errors="coerce").fillna(0).astype(int)
    return df

patient_demo = load_patient(psv_files[0])
display(patient_demo[AVAILABLE_FEATURES + ["SepsisLabel"]].head())


## 6. Identify Sepsis Onset

For this project, the first hourly record with `SepsisLabel == 1` is treated as the dataset's observed sepsis-onset label position. If a patient never has a positive label, the patient is treated as non-septic.

This is a project preprocessing convention based on the provided label; it must not be interpreted as independently deriving clinical sepsis onset.

In [ ]:
def get_sepsis_onset(df: pd.DataFrame):
    positive_positions = np.flatnonzero(df["SepsisLabel"].to_numpy() == 1)
    return int(positive_positions[0]) if len(positive_positions) else None

onset_demo = get_sepsis_onset(patient_demo)
print("Observed sepsis onset row:", onset_demo)


## 7. Discretization Strategy

Sequential pattern mining requires symbolic events rather than raw continuous values. For the first implementation, each numeric variable is converted into LOW/NORMAL/HIGH states using patient-independent quantile thresholds estimated from the selected training set.

The thresholds are estimated once from the selected training set and then reused. This prevents each patient from receiving a different definition of HIGH or LOW.

In [ ]:
def estimate_quantile_thresholds(files, features, max_files=None):
    """Estimate global 33rd/67th percentile thresholds from the selected training set."""
    selected_files = files if max_files is None else files[:max_files]
    chunks = []
    for path in selected_files:
        df = pd.read_csv(path, sep="|", usecols=lambda c: c in features)
        chunks.append(df)
    combined = pd.concat(chunks, ignore_index=True)
    thresholds = {}
    for col in features:
        values = pd.to_numeric(combined[col], errors="coerce").dropna()
        if len(values) == 0:
            thresholds[col] = (np.nan, np.nan)
        else:
            thresholds[col] = (values.quantile(1/3), values.quantile(2/3))
    return thresholds

# For development, you can temporarily set MAX_THRESHOLD_FILES to a smaller value.
# Set to None for the selected training set.
MAX_THRESHOLD_FILES = None
thresholds = estimate_quantile_thresholds(psv_files, AVAILABLE_FEATURES, MAX_THRESHOLD_FILES)

threshold_table = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in thresholds.items()],
    columns=["Feature", "Low_Upper_Quantile", "High_Lower_Quantile"]
)
display(threshold_table)


In [ ]:
def discretize_value(value, low_threshold, high_threshold):
    if pd.isna(value):
        return None
    if value < low_threshold:
        return "LOW"
    if value > high_threshold:
        return "HIGH"
    return "NORMAL"

def row_to_events(row, features, thresholds):
    events = []
    for feature in features:
        low, high = thresholds[feature]
        if pd.isna(low) or pd.isna(high):
            continue
        state = discretize_value(row[feature], low, high)
        if state is not None:
            events.append(f"{feature}_{state}")
    return events

print(row_to_events(patient_demo.iloc[0], AVAILABLE_FEATURES, thresholds))


## 8. Build Positive Pre-Sepsis Sequences

A sequence is represented as a list of hourly event sets. Each hourly item contains the observed symbolic states for that hour.

Only observations before the first positive `SepsisLabel` are included in the positive window.

In [ ]:
def build_positive_sequence(df, features, thresholds, window_hours):
    onset = get_sepsis_onset(df)
    if onset is None or onset < window_hours:
        return None
    window = df.iloc[onset - window_hours:onset]
    sequence = []
    for _, row in window.iterrows():
        events = row_to_events(row, features, thresholds)
        if events:
            sequence.append(events)
    return sequence if sequence else None


## 9. Build Negative Comparison Sequences

For non-septic patients, construct an equivalent-length window. The end position is sampled from the patient's available timeline while ensuring enough observations exist for the requested window.

In [ ]:
def build_negative_sequence(df, features, thresholds, window_hours, rng):
    if len(df) < window_hours:
        return None
    max_end = len(df)
    end = int(rng.integers(window_hours, max_end + 1))
    window = df.iloc[end - window_hours:end]
    sequence = []
    for _, row in window.iterrows():
        events = row_to_events(row, features, thresholds)
        if events:
            sequence.append(events)
    return sequence if sequence else None


## 10. Process the Selected Training Set

This creates the positive and negative sequence collections. The patient identifier is retained as metadata so later notebooks can enforce patient-level separation.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
positive_sequences = []
negative_sequences = []
patient_metadata = []

for i, path in enumerate(psv_files, start=1):
    df = load_patient(path)
    patient_id = path.stem
    onset = get_sepsis_onset(df)

    if onset is not None:
        sequence = build_positive_sequence(
            df, AVAILABLE_FEATURES, thresholds, PRE_SEPSIS_HOURS
        )
        if sequence is not None:
            positive_sequences.append({
                "patient_id": patient_id,
                "sequence": sequence,
                "onset_row": onset,
                "window_hours": PRE_SEPSIS_HOURS,
                "label": 1,
            })
    else:
        sequence = build_negative_sequence(
            df, AVAILABLE_FEATURES, thresholds, NEGATIVE_WINDOW_HOURS, rng
        )
        if sequence is not None:
            negative_sequences.append({
                "patient_id": patient_id,
                "sequence": sequence,
                "onset_row": None,
                "window_hours": NEGATIVE_WINDOW_HOURS,
                "label": 0,
            })

    if i % 500 == 0 or i == len(psv_files):
        print(f"Processed {i:,}/{len(psv_files):,} patients")

print(f"Positive sequences: {len(positive_sequences):,}")
print(f"Negative sequences: {len(negative_sequences):,}")


## 11. Inspect the Generated Sequences

In [ ]:
if positive_sequences:
    example = positive_sequences[0]
    print("Patient:", example["patient_id"])
    print("Observed onset row:", example["onset_row"])
    print("Number of symbolic time steps:", len(example["sequence"]))
    for hour, events in enumerate(example["sequence"], start=1):
        print(f"Hour -{PRE_SEPSIS_HOURS - hour + 1}: {events}")
else:
    print("No positive sequences were generated with the current window size.")
    
if negative_sequences:
    example = negative_sequences[0]
    print("\nNegative patient:", example["patient_id"])
    print("Number of symbolic time steps:", len(example["sequence"]))
    for hour, events in enumerate(example["sequence"], start=1):
        print(f"Hour {hour}: {events}")


## 12. Save Outputs for Notebook 02

The next notebook will consume these serialized sequence files. Saving them avoids repeatedly reading and preprocessing every PSV file.

In [ ]:
positive_path = OUTPUT_DIR / "positive_sequences.pkl"
negative_path = OUTPUT_DIR / "negative_sequences.pkl"
threshold_path = OUTPUT_DIR / "discretization_thresholds.pkl"
metadata_path = OUTPUT_DIR / "sequence_metadata.csv"

with open(positive_path, "wb") as f:
    pickle.dump(positive_sequences, f)
with open(negative_path, "wb") as f:
    pickle.dump(negative_sequences, f)
with open(threshold_path, "wb") as f:
    pickle.dump(thresholds, f)

metadata_rows = []
for item in positive_sequences + negative_sequences:
    metadata_rows.append({
        "patient_id": item["patient_id"],
        "label": item["label"],
        "onset_row": item["onset_row"],
        "window_hours": item["window_hours"],
        "sequence_length": len(item["sequence"]),
    })

pd.DataFrame(metadata_rows).to_csv(metadata_path, index=False)

print("Saved:")
for path in [positive_path, negative_path, threshold_path, metadata_path]:
    print(" -", path)


## 13. Sanity Checks

Before moving to PrefixSpan, verify that the generated data contain both cohorts and that no post-onset observations were included in positive windows.

In [ ]:
assert len(positive_sequences) > 0, "No positive sequences were generated."
assert len(negative_sequences) > 0, "No negative sequences were generated."
assert all(item["label"] == 1 for item in positive_sequences)
assert all(item["label"] == 0 for item in negative_sequences)
assert all(item["onset_row"] is not None for item in positive_sequences)
assert all(len(item["sequence"]) <= PRE_SEPSIS_HOURS for item in positive_sequences)

print("All basic sequence-construction sanity checks passed.")
print(f"Final positive sequences: {len(positive_sequences):,}")
print(f"Final negative sequences: {len(negative_sequences):,}")
